### 1.4.9.3. Numerical Methods for Systems and Higher-Order Equations

$$
\mathbf{X}' = \mathbf{F}(t, \mathbf{X}), \qquad
\mathbf{X}_{n+1} = \mathbf{X}_n + h\,\mathbf{F}(t_n, \mathbf{X}_n) \ (\text{Euler}),
\quad \text{RK4 applied componentwise} .
$$

**Explanation:**

Euler and Runge–Kutta extend unchanged to **systems**: write the problem in the first-order [vector form](../08_Systems_of_Linear_First_Order_ODEs/01_linear_systems_in_matrix_form.ipynb) $\mathbf{X}' = \mathbf{F}(t, \mathbf{X})$ and apply the same step formulas with $\mathbf{X}$ and $\mathbf{F}$ as vectors. A higher-order equation is handled by first reducing it to a system (state variables = the function and its derivatives). This vectorized [RK4](./02_runge_kutta_methods.ipynb) is exactly how trajectory simulators, reinforcement-learning environments, and differentiable ODE solvers integrate state dynamics forward in time.

**Properties:**
- An $n$th-order IVP becomes an $n$-dimensional first-order system, then one stepper integrates all components together.
- Library integrators (`scipy.integrate.solve_ivp`, CasADi integrators) implement adaptive versions of these formulas.

**Numerical Example:**

Solve the harmonic oscillator $x'' + x = 0$, $x(0) = 1$, $x'(0) = 0$ (exact $x = \cos t$) with RK4, $h = 0.1$.

**Reduce to a system.** Let $x_1 = x$, $x_2 = x'$. Then

$$
\mathbf{X}' = \begin{pmatrix} x_1' \\ x_2' \end{pmatrix} = \begin{pmatrix} x_2 \\ -x_1 \end{pmatrix} = \mathbf{F}(\mathbf{X}), \qquad \mathbf{X}(0) = \begin{pmatrix} 1 \\ 0 \end{pmatrix}.
$$

**One RK4 step** ($\mathbf{X}_0 = (1, 0)$): $\mathbf{k}_1 = \mathbf{F}(1,0) = (0, -1)$; $\mathbf{k}_2 = \mathbf{F}\big((1,0) + 0.05\,\mathbf{k}_1\big) = \mathbf{F}(1, -0.05) = (-0.05, -1)$; similarly $\mathbf{k}_3 = (-0.05, -0.9975)$, $\mathbf{k}_4 = (-0.0999, -0.995)$, giving

$$
\mathbf{X}_1 = \mathbf{X}_0 + \tfrac{0.1}{6}(\mathbf{k}_1 + 2\mathbf{k}_2 + 2\mathbf{k}_3 + \mathbf{k}_4) \approx (0.99500,\, -0.09983) .
$$

This matches $(\cos 0.1, -\sin 0.1) = (0.99500, -0.09983)$. Stepping to $t = 2\pi$ reproduces $\cos t$ to high accuracy.

In [1]:
import numpy as np

def rk4_system(field, state0, step, steps):
    states = [np.array(state0, dtype=float)]
    for _ in range(steps):
        state = states[-1]
        k1 = field(state)
        k2 = field(state + step / 2 * k1)
        k3 = field(state + step / 2 * k2)
        k4 = field(state + step * k3)
        states.append(state + step / 6 * (k1 + 2 * k2 + 2 * k3 + k4))
    return np.array(states)

oscillator = lambda state: np.array([state[1], -state[0]])
trajectory = rk4_system(oscillator, [1.0, 0.0], 0.1, 63)
times = np.arange(trajectory.shape[0]) * 0.1

print("first RK4 step X_1 =", np.round(trajectory[1], 5), " exact (cos0.1, -sin0.1) =",
      np.round([np.cos(0.1), -np.sin(0.1)], 5))
print("max error in x over [0, 2pi]:", np.max(np.abs(trajectory[:, 0] - np.cos(times))))

first RK4 step X_1 = [ 0.995   -0.09983]  exact (cos0.1, -sin0.1) = [ 0.995   -0.09983]
max error in x over [0, 2pi]: 4.0796291235767335e-06


**Equivalent SciPy implementation:**

`scipy.integrate.solve_ivp` integrates the same vector field with an adaptive Runge–Kutta scheme, matching the from-scratch result; CasADi integrators provide the control-oriented analogue.

In [2]:
import numpy as np
from scipy.integrate import solve_ivp

solution = solve_ivp(lambda t, state: [state[1], -state[0]], (0, 2 * np.pi), [1.0, 0.0],
                     t_eval=np.linspace(0, 2 * np.pi, 64), rtol=1e-8)

print("scipy x(2pi) =", round(solution.y[0, -1], 6), " exact cos(2pi) =", round(np.cos(2 * np.pi), 6))

scipy x(2pi) = 0.999998  exact cos(2pi) = 1.0


**References:**

[📘 Zill, D. G. (2016). *A First Course in Differential Equations with Modeling Applications* (11th ed.). Cengage Learning.](https://www.cengage.com/c/a-first-course-in-differential-equations-with-modeling-applications-11e-zill/9781305965720/)

---

[⬅️ Previous: Runge-Kutta Methods](./02_runge_kutta_methods.ipynb)